In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pylab import rcParams
random_state = 42

sys.path.append(os.path.abspath(".."))

import python_code.Scripts2 as sc
import python_code.Reference as ref
import pickle

from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,recall_score,precision_score
from treeinterpreter import treeinterpreter as ti
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier,ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier,BaggingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

print("Everything imported successfully")

Everything imported successfully


In [2]:
df = pd.read_csv("../data/final.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3904 entries, 0 to 3903
Data columns (total 33 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   FSASSET   3904 non-null   float64
 1   FSEARN    3904 non-null   float64
 2   WRK_POOR  3904 non-null   float64
 3   FSTOTDE2  3904 non-null   float64
 4   TPOV      3904 non-null   float64
 5   FSTANF    3904 non-null   float64
 6   FSNETINC  3904 non-null   float64
 7   FSSLTDED  3904 non-null   float64
 8   FSUSIZE   3904 non-null   float64
 9   FSTOTDED  3904 non-null   float64
 10  FSERNDE2  3904 non-null   float64
 11  FSGRINC   3904 non-null   float64
 12  FSWAGES   3904 non-null   float64
 13  RAWERND   3904 non-null   float64
 14  FSGA      3904 non-null   float64
 15  FSDIS     3904 non-null   float64
 16  SHELDED   3904 non-null   float64
 17  FSERNDED  3904 non-null   float64
 18  TANF_IND  3904 non-null   float64
 19  FSNELDER  3904 non-null   float64
 20  FSSLTDE2  3904 non-null   float64
 21  FS

##Data Prep
#capital x, lowercase y

In [4]:
X = df.drop(columns = ['CAT_ELIG'])
y = df['CAT_ELIG']

In [5]:
#baseline, NULL Model
y.value_counts(normalize=True)

CAT_ELIG
1.0    0.662398
0.0    0.337602
Name: proportion, dtype: float64

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.3,random_state=random_state)

In [7]:
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

In [8]:
np.savetxt('../data/TrainTest/X_train.csv',X_train,delimiter=',')
np.savetxt('../data/TrainTest/y_train.csv',y_train,delimiter=',')
np.savetxt('../data/TrainTest/X_test.csv',X_test,delimiter=',')
np.savetxt('../data/TrainTest/y_test.csv',y_test,delimiter=',')

In [9]:
pca = PCA(n_components=10,random_state=42)
pca.fit(X_train)
X_train_pc = pca.transform(X_train)
X_test_pc = pca.transform(X_test)

## Models

In [10]:
models = {
    'LogReg': LogisticRegression(),
    'Decision Tree':DecisionTreeClassifier(),
    'Random Forest':RandomForestClassifier(),
    'Gradient Boost':GradientBoostingClassifier(),
    'Ada Boost':AdaBoostClassifier(),
    'SVC':SVC(),
    'Naive Bayes':GaussianNB()}

In [11]:
#adapted from Dan Brown lecture
final = pd.DataFrame(columns = ['cross_val_train','cross_val_test','test_recall','test_precision'])
idx=0
while idx < len(models.keys()):
    for name,model in models.items():
        results = {}
        results['name']=name
        name=model.fit(X_train, y_train)
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        results['cross_val_train'] = np.mean(cross_val_score(model,X_train,y_train,cv=4))
        results['cross_val_test'] = np.mean(cross_val_score(model,X_test,y_test,cv=4))
        results['test_recall'] = recall_score(y_test, y_pred_test)
        results['test_precision'] = precision_score(y_test, y_pred_test)
        final = pd.concat([final, pd.DataFrame([results])], ignore_index=True)
        idx+=1

In [12]:
final.set_index('name')

,cross_val_train,cross_val_test,test_recall,test_precision
name,,,,
LogReg,0.884334,0.862628,0.900262,0.903821
Decision Tree,0.909956,0.917235,0.930446,0.940318
Random Forest,0.940337,0.931741,0.965879,0.955844
Gradient Boost,0.940337,0.933447,0.950131,0.961487
Ada Boost,0.879209,0.882253,0.910761,0.906005
SVC,0.8847,0.854949,0.901575,0.928378
Naive Bayes,0.767204,0.741468,0.833333,0.908441


In [13]:
pc_final = pd.DataFrame(columns = ['cross_val_train','cross_val_test','test_recall','test_precision'])
idx=0
while idx < len(models.keys()):
    for name,model in models.items():
        results = {}
        results['name']=name
        name=model.fit(X_train_pc, y_train)
        y_pred_train = model.predict(X_train_pc)
        y_pred_test = model.predict(X_test_pc)
        results['cross_val_train'] = np.mean(cross_val_score(model,X_train_pc,y_train,cv=4))
        results['cross_val_test'] = np.mean(cross_val_score(model,X_test_pc,y_test,cv=4))
        results['test_recall'] = recall_score(y_test, y_pred_test)
        results['test_precision'] = precision_score(y_test, y_pred_test)
        pc_final = pd.concat([pc_final, pd.DataFrame([results])], ignore_index=True)
        idx+=1

In [14]:
pc_final.set_index('name')

,cross_val_train,cross_val_test,test_recall,test_precision
name,,,,
LogReg,0.783675,0.763652,0.892388,0.770102
Decision Tree,0.832723,0.816553,0.893701,0.871959
Random Forest,0.883602,0.855802,0.933071,0.901141
Gradient Boost,0.877745,0.860068,0.908136,0.882653
Ada Boost,0.838214,0.822526,0.885827,0.858779
SVC,0.831991,0.834471,0.874016,0.871728
Naive Bayes,0.72694,0.713311,0.958005,0.6926


## Model Selection

In [15]:
for name,model in models.items():
    name = model.fit(X_train,y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    print(name)
    print(classification_report(y_test, y_pred_test))
    print('\n')

LogisticRegression()
              precision    recall  f1-score   support

         0.0       0.82      0.82      0.82       410
         1.0       0.90      0.90      0.90       762

    accuracy                           0.87      1172
   macro avg       0.86      0.86      0.86      1172
weighted avg       0.87      0.87      0.87      1172



DecisionTreeClassifier()
              precision    recall  f1-score   support

         0.0       0.87      0.89      0.88       410
         1.0       0.94      0.93      0.94       762

    accuracy                           0.92      1172
   macro avg       0.91      0.91      0.91      1172
weighted avg       0.92      0.92      0.92      1172



RandomForestClassifier()
              precision    recall  f1-score   support

         0.0       0.93      0.92      0.92       410
         1.0       0.96      0.96      0.96       762

    accuracy                           0.95      1172
   macro avg       0.94      0.94      0.94      1172

## Running Models

In [22]:
rf = RandomForestClassifier()
params={'max_depth':[None, 3, 4],
       'max_features':[None, 'sqrt'], # Changed 'auto' to 'sqrt'
       'n_estimators':[75, 100, 125]}
rf_gs = GridSearchCV(rf,param_grid=params)
rf_gs.fit(X_train,y_train)
print(rf_gs.best_score_)
rf_gs.best_params_

0.9443605145616114


{'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 125}

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

In [ ]:
instances = X_test[[735]]
instances

In [ ]:
ft_list = []
prediction, bias, contributions = ti.predict(rf, instances)
print( "Prediction", prediction)
print( "Bias (trainset prior)", bias)
print ("Feature contributions:")
for c, feature in zip(contributions[0], 
                             X.columns):
    ft_list.append((feature, np.round(c, 2)))
    print (feature, c)
    
labels, values = zip(*ft_list)

In [ ]:
df1 = pd.DataFrame(ft_list,columns=['feature','array'])
df2 = pd.DataFrame(df1["array"].to_list(), columns=['pred_0', 'pred_1'])
coef_df = pd.concat([df1,df2],axis=1).drop(columns=['array'])
coef_df.to_csv('../data/2018_indicators/coef.csv',index=None)

In [ ]:
fig, ax = plt.subplots(figsize=(20,10))
plt.title('Random Forest range of Coefficients Effect on SNAP \n (the larger the range, the more impact on predicting SNAP)')
plt.grid(zorder=0,alpha = 0.2)
xs = np.arange(len(labels))
ax.bar(xs,coef_df['pred_0'], label = 'pred 0')
ax.bar(xs,coef_df['pred_1'],label = 'pred 1')
ax.axhline(y=0, linestyle='--', color='black', linewidth=4)
ax.set_xticks(coef_df.index)
ax.set_xticklabels(coef_df['feature'],rotation = 45)
plt.legend()
plt.savefig('../images/rf_corr.png');

In [ ]:
et = ExtraTreeClassifier()
params={'max_depth':[None,3,4],
       'max_features':[None,'sqrt'],
       'max_leaf_nodes':[5,10]}
et_gs = GridSearchCV(et,param_grid=params)
et_gs.fit(X_train,y_train)
print(et_gs.best_score_)
et_gs.best_params_

In [ ]:
bag = BaggingClassifier()
bag.fit(X_train,y_train)
y_pred_train = bag.predict(X_train)
y_pred_test = bag.predict(X_test)
print(f'cross_val_train = {np.mean(cross_val_score(model,X_train_pc,y_train,cv=4))}')
print(f'cross_val_test = {np.mean(cross_val_score(model,X_test_pc,y_test,cv=4))}')
print(f'test_recall = {recall_score(y_test, y_pred_test)}')
print(f'test_precision = {precision_score(y_test, y_pred_test)}')

In [ ]:
## Final Model

In [ ]:
vote = VotingClassifier([
    ('rf',RandomForestClassifier(bootstrap=False,n_estimators=1000)),
    ('gb',GradientBoostingClassifier(max_depth=10,subsample=0.8)),
    ('bag',BaggingClassifier(n_estimators = 10))
])

In [ ]:
vote.fit(X_train,y_train)

In [ ]:
filename = '../data/final_model.sav'
pickle.dump(vote, open(filename, 'wb'))